<a href="https://colab.research.google.com/github/rcsb/rcsb-training-resources/blob/master/training-events/2026/python-rcsb-api/advanced_search_and_data_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install `rcsb-api`
%pip install --upgrade rcsb-api

# Advanced Search and Data API Usage

Building upon existing documentation and tutorials for utilizing the Search and Data API modules of the `rcsb-api` package, this notebook is intended to provide some additional, more advanced use cases that will help you take full advantage of all the tooling and options available.

Specifically, this tutorial will cover:
1. Performing structure-similarity searches with the [new AI-powered 3D search service](https://www.rcsb.org/news/feature/69bc5e328557ae0f261bc71c)
2. Grouping of search results by sequence identity
3. Constructing queries containing nested search attributes
4. Fine-tuning of `rcsb-api` package configurations for optimal performance
5. Change in Data API usage in Jupyter running Python 3.14+

## 1. Performing structure-similarity searches with the new AI-powered 3D search service
RCSB.org recently introduced a [new AI-powered 3D search service](https://www.rcsb.org/news/feature/69bc5e328557ae0f261bc71c) service, which can perform archive wide similarity searches at great speed.

Along with this release also introduced several changes in the parameters provided to the [`StructSimilarityQuery`](https://rcsbapi.readthedocs.io/en/latest/search_api/query_construction.html#structure-similarity-search). Specifically, the updated version of the class adds the following parameters: `number_of_candidates`, `ptmscore_cutoff`, and `similarity_type`. For details on all available arguments, see the [documentation](https://rcsbapi.readthedocs.io/en/latest/search_api/query_construction.html#structure-similarity-search).

Below is an example of searching for structures of similar 3D similarity to PDB ID 4HHB:

In [ ]:
from rcsbapi.search import StructSimilarityQuery

q1 = StructSimilarityQuery(
    structure_search_type="entry_id",
    entry_id="4HHB",
    assembly_id="1",
    similarity_type="local",
    target_search_space="assembly"
)

# Return assembly IDs
for rid in q1("assembly"):
    print(rid)

You can also provide your own local structure files for performing archive-wide searches, either by providing the local file path or a URL:

In [ ]:
from rcsbapi.search import StructSimilarityQuery

# Using `file_path`
q2 = StructSimilarityQuery(
    structure_search_type="file_upload",
    file_path="/PATH/TO/FILE.cif",  # specify local model file path
    file_format="cif"
)
list(q2())


# Using file_url
q3 = StructSimilarityQuery(
    structure_search_type="file_url",
    file_url="https://files.rcsb.org/download/4HHB.cif",
    file_format="cif"
)
list(q3())

## 2. Grouping of search results by sequence identity
Search API results can be [grouped by various strategies](https://rcsbapi.readthedocs.io/en/latest/search_api/additional_examples.html#groupby-example), such as by sequence identity or UniProt accession code, in order to consolidate the total number of returned results and obtain a "representative" structure of highly-similar entities.

To demonstrate this, let's assume the following use-case. We want to:
1) Group all protein entities together by sequence identity of 90% and return a list of just the representative structure of each group
2) And then retrieve the associated 1-letter sequence and Enzyme Commision (EC) number of all represntatives as well as the cluster ID


For step (1), we perform a Search API query with the `group_by` (and input `GroupBy`) argument, as follows:

In [ ]:
from rcsbapi.search import search_attributes as attrs
from rcsbapi.search import GroupBy

query = attrs.entity_poly.rcsb_entity_polymer_type == "Protein"

# Note that this will take several minutes to complete due to the breadth of the search
search_results = list(
    query(
        group_by=GroupBy(
            aggregation_method="sequence_identity",
            similarity_cutoff=90,
        ),
        group_by_return_type="representatives",
        return_type="polymer_entity",
    )
)

print(len(search_results))
print(search_results[0:5])  # will contain representative entities like ['104M_1', '10AF_1', '10BM_1', '10FT_2', '10GH_3']

The "representatives" returned above will basically just be one member of the group, with no particular preference of one or the other. If specifically the best resolution structure is desired, that is possible, but it will slow down the query significantly (especially for a query of this breadth as done here, as it is fetching every protein in the PDB). If encounter any 504 errors, please be patient and try again a few minutes later; if it persists upon multiple retries, please report it to [info@rcsb.org](info@rcsb.org).

Next, for step (2), the data retrieval can be done as follows:

In [ ]:
from rcsbapi.data import DataQuery as Query

# Constructing the Query object
query = Query(
    input_type="polymer_entities",
    input_ids=search_results,
    return_data_list=[
        # Both `cluster_id` and `identity` are needed to determine the correct cluster ID
        "rcsb_cluster_membership.cluster_id",
        "rcsb_cluster_membership.identity",
        #
        "entity_poly.pdbx_seq_one_letter_code_can",  # 1-letter sequence
        "rcsb_polymer_entity.rcsb_enzyme_class_combined",  # EC number
    ]
)

# Executing the query
data_results = query.exec()
print(len(data_results["data"]["polymer_entities"]))

This will return the requested set of data for the representatives. Note that the returned set of `rcsb_cluster_membership` data will contain the `cluster_id`s for all `identity` levels, so it will be necessary to post-process the data to extract out the correct `cluster_id` for the `identity` of `90`.

## 3. Constructing queries containing nested search attributes
This third topic is primarily intended to serve as an update/announcment on the usage of the Search API module, specifically in the construction of attribute-based queries.

Certain attributes in the RCSB.org schema are "coupled"—that is, they come in pairs that should always be provided together. For example, the attribute `rcsb_binding_affinity.value` should always be paired with the attribute `rcsb_binding_affinity.type`, since the `value` by itself is meaningless without specifying the `type`. Importantly, if these attributes are not provided properly in the construction of a search query, it can lead to incorrect results.

Accordingly, to properly handle these types of "[nested attributes](https://rcsbapi.readthedocs.io/en/latest/search_api/query_construction.html#nested-attributes)", a new class was introduced, `NestedAttributeQuery`, which enables the forced nesting (i.e., grouping) of the two attributes toegether in query construction. 

Let's take a look at an example, in which we attempt to construct a query containing nested attributes without properly nesting them:

In [ ]:
from rcsbapi.search import AttributeQuery

# q1 and q2 represent nested attributes
q1 = AttributeQuery("rcsb_binding_affinity.type", "exact_match", "EC50")
q2 = AttributeQuery("rcsb_binding_affinity.value", "equals", 2.0)

# q3 and q4 are unrelated attributes that we want to include as well
q3 = AttributeQuery("rcsb_entry_info.selected_polymer_entity_types", "exists")
q4 = AttributeQuery("rcsb_nonpolymer_entity_container_identifiers.nonpolymer_comp_id", "exists")

# combine the sub-queries with AND
query = q1 & q2 & q3 & q4

print("Number of results for un-nested query: ", len(list(query())))

As you can see, if you ever forget to nest the attributes together, several `WARNING` messages will be printed informing you on how to properly nest them. The query will still run though, but will likely return an incorrect list of IDs (in this case, 182 results at the time of writing).

Now, let's heed that message and construct the query properly:

In [ ]:
from rcsbapi.search import AttributeQuery, NestedAttributeQuery

# q1 and q2 represent nested attributes
q1 = AttributeQuery("rcsb_binding_affinity.type", "exact_match", "EC50")
q2 = AttributeQuery("rcsb_binding_affinity.value", "equals", 2.0)

# q3 and q4 are unrelated attributes that we want to include as well
q3 = AttributeQuery("rcsb_entry_info.selected_polymer_entity_types", "exists")
q4 = AttributeQuery("rcsb_nonpolymer_entity_container_identifiers.nonpolymer_comp_id", "exists")

# First combine the nested attributes together explicitly, then combine all sub-queries with AND
nested_query = NestedAttributeQuery(q1, q2) & q3 & q4

print("Number of results for nested query: ", len(list(nested_query())))

We now receive a clean response reporting 99 total matching IDs (at the time of writing). Clearly, it is critical to use this `NestedAttributeQuery` whenever necessary to ensure correct results are returned.

## 4. Fine-tuning of `rcsb-api` package configurations for optimal performance
A variety of [configuration options exist](https://rcsbapi.readthedocs.io/en/latest/config/custom_configuration.html) for the `rcsb-api` package that you can adjust to control its performance, such as the number of requests per second, the API timeout, as well as the input ID limit for certain modules.

These are especially useful for fine-tuning your scripts to improve the throughput (or, vice-versa, reduce the aggressiveness in the event of timeouts or other issues).

For example, to increase the number of concurrent Data API requests to run, you can do this:

In [ ]:
from rcsbapi.config import config

# Override the default number of concurrent Data API requests
config.DATA_API_MAX_CONCURRENT_REQUESTS = 8

Please note of course to adjust these settings with care and caution, as attempting to increase them too much may end up hindering performance by exceeding RCSB.org API rate limits or server timeouts.

## 5. Change in Data API usage in Jupyter running Python 3.14+
When working in Jupyter environments *with* Python 3.14+, execution of Data API queries (`DataQuery`) must be explicitly `awaited` ([see details](https://rcsbapi.readthedocs.io/en/latest/data_api/quickstart.html#important-changes-to-jupyter-behavior-in-python-3-14))
  - This addresses an issue with the `async` behavior of Jupyter environments coupled with changes to `asyncio` in Python 3.14
  - This change **does not** impact code run in standard Python scripts (of any Python version); it only affects code run in Jupyter that uses Python 3.14 or greater

If you open up a Jupyter notebook with Python 3.14+, a `WARNING` message will be printed informing you of this new style of usage.

For example, you will be instructed to execute `DataQuery` queries as follows:

In [ ]:
from rcsbapi.data import DataQuery

query = DataQuery(input_type="entries", input_ids=[...], return_data_list=[...])

results = await query.exec()  # You will need to explicitly `await` the query execution call

#### Again, this change ***does not impact*** code run in standard Python scripts (of any Python version); it only affects code run in Jupyter that uses Python 3.14 or greater.

